# Дообучение детектора зданий (geoqc)

Универсальный ноутбук (`DECISIONS.md D-004`): один и тот же файл запускается **локально** (CUDA / Apple MPS / CPU)
и в **Google Colab**. Ячейки, нужные только в Colab, помечены `# COLAB ONLY` и завёрнуты в `if IN_COLAB:` — локально они
пропускаются.

Что делает ноутбук:

1. определяет окружение и устройство, фиксирует seed;
2. (Colab) монтирует Drive, ставит пакеты, распаковывает `bundle.zip` из `geoqc cv bundle`;
3. готовит датасет `data/interim/dataset` из разметки **val** (`geoqc cv export-dataset`): внутри val — сплит `train`/`holdout` 80/20;
4. дообучает YOLOv8-seg (Ultralytics) **или** U-Net (segmentation_models_pytorch): AMP, кэширование, чекпоинт каждой эпохи,
   resume, ранняя остановка, лог в CSV;
5. экспортирует лучшие веса в `models/`, считает sha256 → `model_version`, обновляет `config/cv.yaml` и `models/MODEL_CARD.md`;
6. подбирает порог уверенности на holdout-части val (`threshold-sweep`).

Разметка тестового сплита в ноутбуке **не используется**: ни для обучения, ни для валидации, ни для подбора порога.
Порядок запуска и ожидаемое время — в `notebooks/README.md`.

## 1. Окружение: Colab или локально, устройство, seed

In [ ]:
# Environment: detect Colab, pick the device (cuda > mps > cpu), fix seeds.
import importlib.util
import os
import platform
import random
import sys
from pathlib import Path

import numpy as np
import torch

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # no "google" namespace package at all (plain local venv)
    IN_COLAB = False
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
os.environ.setdefault("PYTHONHASHSEED", str(SEED))

print(
    f"IN_COLAB={IN_COLAB} device={DEVICE} torch={torch.__version__} python={platform.python_version()}"
)
print(f"cuda={torch.cuda.is_available()} mps={torch.backends.mps.is_available()}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

## 2. COLAB ONLY: Drive, пакеты, распаковка `bundle.zip`

Локально шаг пропускается. В Colab положите `data/interim/bundle.zip` (результат `make bundle` / `geoqc cv bundle`)
на Google Drive и укажите путь в `BUNDLE_ZIP`. Архив содержит тайлы, разметку val, `tiles_split.csv`, `config/`, исходники
пакета `geoqc` — тестовая разметка в архив не попадает по построению (`geoqc.cv.bundle`).

In [ ]:
# COLAB ONLY: mount Google Drive, install dependencies, unpack the bundle produced by `geoqc cv bundle`.
BUNDLE_ZIP = "/content/drive/MyDrive/geoqc/bundle.zip"  # adjust to your Drive layout
COLAB_ROOT = Path("/content/geoqc")
COLAB_PACKAGES = [
    "ultralytics>=8.3",
    "segmentation-models-pytorch>=0.4",
    "albumentations>=1.4",
    "rasterio>=1.3.10",
    "geopandas>=1.0",
    "pyogrio>=0.9",
    "shapely>=2.0",
    "opencv-python-headless>=4.10",
    "pydantic-settings>=2.4",
    "ruamel.yaml>=0.18",
    "typer>=0.12",
    "pyyaml>=6.0",
]
if IN_COLAB:
    import subprocess
    import zipfile

    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *COLAB_PACKAGES], check=True)
    COLAB_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_ZIP) as zf:
        zf.extractall(COLAB_ROOT)
    print("bundle extracted to", COLAB_ROOT)

## 3. Пути и конфигурация

Локально корень репозитория ищется вверх от текущего каталога по `pyproject.toml`; в Colab это каталог распаковки.
Пакет `geoqc` импортируется из `src/` (в Colab — из архива), конфиг читается `geoqc.config.load_config()`.

In [ ]:
# Resolve the repository root and load config/*.yaml through geoqc.config.
import yaml  # noqa: E402

if IN_COLAB:
    REPO_ROOT = COLAB_ROOT
else:
    REPO_ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
    )
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)
os.environ["GEOQC_CONFIG_DIR"] = str(REPO_ROOT / "config")

from geoqc.config import load_config  # noqa: E402

CFG = load_config(REPO_ROOT / "config")
CV = CFG["cv"]
DATASET_DIR = REPO_ROOT / CV["dataset"]["out_dir"]
MODELS_DIR = REPO_ROOT / "models"
RUNS_DIR = MODELS_DIR / "runs"  # models/* is git-ignored except MODEL_CARD.md
REPORTS_DIR = REPO_ROOT / "reports"
for d in (MODELS_DIR, RUNS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("REPO_ROOT =", REPO_ROOT)
print("backend =", CV["backend"], "| model_version =", CV.get("model_version"))
print(yaml.safe_dump({"dataset": CV["dataset"], "vectorize": CV["vectorize"]}, sort_keys=False))

## 4. Данные: разметка val и/или SpaceNet 2

`DATA_SOURCE` выбирает материал для обучения:

* `"val"` — наша разметка: `export_dataset` читает `data/labels/val/buildings.geojson` (только `verified=true`), режет
  полигоны по тайлам, пишет `images/{train,holdout}/*.png`, YOLO-seg `labels/*.txt`, бинарные `masks/*.png`, `data.yaml`,
  `split.csv`, `meta.json`. Сплит train/holdout детерминирован (`dataset.seed`) и стратифицирован по `stratum`;
* `"spacenet"` — только SpaceNet 2 (CC BY-SA 4.0, `geoqc cv fetch-external` + `prepare-external`, `docs/datasets/spacenet2.md`):
  проверка всего пути обучения, пока разметка val не готова. Метрики на holdout SpaceNet — **не метрики работы**,
  `config/cv.yaml` в этом режиме не меняется, подбор порога пропускается;
* `"val+spacenet"` — train-часть SpaceNet добавляется к train-части val; holdout и порог — только по val.

Если SpaceNet ещё не подготовлен, ячейка скачает и подготовит его (~800 МБ при 100 тайлах на город).
Если датасет val уже экспортирован (например, приехал в `bundle.zip`), экспорт пропускается; `path` в `data.yaml`
переписывается под текущую машину.

In [ ]:
# Training material: our verified val labels, SpaceNet 2, or both (see the markdown above).
from geoqc.cv.dataset import export_dataset, read_holdout_tile_ids  # noqa: E402
from geoqc.cv.external import external_config, fetch_spacenet, prepare_spacenet  # noqa: E402

DATA_SOURCE = "val"  # "val" | "spacenet" | "val+spacenet"
assert DATA_SOURCE in ("val", "spacenet", "val+spacenet"), DATA_SOURCE
EXT_DIR = REPO_ROOT / external_config(CV)["out_dir"]
if DATA_SOURCE != "val" and not (EXT_DIR / "data.yaml").exists():
    print(fetch_spacenet(CV))
    print(prepare_spacenet(CV))

if DATA_SOURCE == "spacenet":
    DATASET_DIR = EXT_DIR
    DATA_YAML = EXT_DIR / "data.yaml"
else:
    DATA_YAML = DATASET_DIR / "data.yaml"
    if not DATA_YAML.exists():
        summary = export_dataset(CV, CFG)
        print(summary.to_dict())
data_cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
data_cfg["path"] = str(DATASET_DIR)  # the bundle may come from another machine
data_cfg["train"] = "images/train"  # reset: a previous run may have appended SpaceNet
if DATA_SOURCE == "val+spacenet":
    data_cfg["train"] = ["images/train", str(EXT_DIR / "images" / "train")]
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False), encoding="utf-8")
TRAIN_DIRS = [DATASET_DIR] + ([EXT_DIR] if DATA_SOURCE == "val+spacenet" else [])
N_TRAIN = sum(len(list((d / "images" / "train").glob("*.png"))) for d in TRAIN_DIRS)
N_HOLDOUT = len(list((DATASET_DIR / "images" / "holdout").glob("*.png")))
print(f"source={DATA_SOURCE} train tiles={N_TRAIN} holdout tiles={N_HOLDOUT}")
print("holdout ids:", read_holdout_tile_ids(DATASET_DIR)[:20])
if N_TRAIN == 0:
    raise RuntimeError(
        "no training tiles: verify the val labels in QGIS and re-run `geoqc cv export-dataset`, "
        'or set DATA_SOURCE = "spacenet" for a pipeline smoke test'
    )

## 5. Опционально: другие внешние датасеты (выключено)

SpaceNet 2 подключается через `DATA_SOURCE` выше. Эта ячейка — для датасетов без загрузчика, например
**INRIA Aerial Image Labeling** (https://project.inria.fr/aerialimagelabeling/ — доступ по регистрации, использование в
исследовательских целях; whitelist плана §9). Только для `TRAINER="yolo"`.
Они не входят в `bundle.zip`; чтобы подмешать их, скачайте данные вручную, положите пары «RGB PNG/TIF + бинарная маска» в
`EXTERNAL_DIR/{images,masks}` и включите `USE_EXTERNAL_DATA`. Ячейка конвертирует маски в YOLO-seg полигоны тем же кодом,
что и основной экспорт, и дописывает каталог в список `train` в `data.yaml`. Доменный сдвиг (другое разрешение, другие крыши)
может ухудшить качество на нашем AOI — проверяйте по holdout.

In [ ]:
# Optional external data without a loader (e.g. INRIA); disabled by default. Masks -> YOLO-seg polygons.
USE_EXTERNAL_DATA = False
EXTERNAL_DIR = (
    REPO_ROOT / "data" / "interim" / "external"
)  # expects images/*.png|tif and masks/*.png
if USE_EXTERNAL_DATA:
    import cv2
    from rasterio.transform import from_origin

    from geoqc.cv.dataset import polygon_to_yolo_lines
    from geoqc.cv.vectorize import mask_to_geometry

    ext_images = EXTERNAL_DIR / "yolo" / "images"
    ext_labels = EXTERNAL_DIR / "yolo" / "labels"
    ext_images.mkdir(parents=True, exist_ok=True)
    ext_labels.mkdir(parents=True, exist_ok=True)
    n_ext = 0
    for mask_path in sorted((EXTERNAL_DIR / "masks").glob("*.png")):
        img_path = next(iter((EXTERNAL_DIR / "images").glob(mask_path.stem + ".*")), None)
        if img_path is None:
            continue
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127
        h, w = mask.shape
        identity = from_origin(0.0, float(h), 1.0, 1.0)  # pixel grid, 1 unit per pixel
        lines: list[str] = []
        geom = mask_to_geometry(mask, identity)
        parts = list(geom.geoms) if hasattr(geom, "geoms") else [geom]
        for part in parts:
            lines.extend(polygon_to_yolo_lines(part, identity, w, h))
        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        cv2.imwrite(str(ext_images / f"{mask_path.stem}.png"), img)
        (ext_labels / f"{mask_path.stem}.txt").write_text(
            "\n".join(lines) + ("\n" if lines else "")
        )
        n_ext += 1
    train_dirs = data_cfg["train"] if isinstance(data_cfg["train"], list) else [data_cfg["train"]]
    data_cfg["train"] = [*train_dirs, str(ext_images)]
    N_TRAIN += n_ext
    DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False), encoding="utf-8")
    print(f"external tiles added: {n_ext}")
else:
    print("external data disabled (USE_EXTERNAL_DATA = False)")

## 6. Параметры обучения

`TRAINER = "yolo"` — дообучение YOLOv8-seg от текущих весов `config/cv.yaml → weights.path` (или от fallback `yolov8m-seg.pt`);
`TRAINER = "unet"` — U-Net из `segmentation_models_pytorch`. `RESUME = True` продолжает прерванный прогон с `last.pt`.
`SMOKE_TEST = True` делает 1 эпоху, чтобы проверить весь путь до экспорта весов.

In [ ]:
# Training hyper-parameters (edit here only).
TRAINER = "yolo"  # "yolo" | "unet"
RUN_NAME = (
    "finetune_yolov8m_seg" if TRAINER == "yolo" else "finetune_unet_resnet34"
) + f"_{DATA_SOURCE.replace('+', '_')}"
SMOKE_TEST = False
EPOCHS = 1 if SMOKE_TEST else 60
PATIENCE = 12
IMGSZ = int(CV.get("imgsz", 512))
BATCH = {"cuda": 8, "mps": 4, "cpu": 2}[DEVICE]
WORKERS = 2 if IN_COLAB else 0
CACHE = "ram" if IN_COLAB else "disk"  # Ultralytics: 'ram' | 'disk' | False
RESUME = False
USE_COMPILE = False  # torch.compile for the U-Net trainer (CUDA only, PyTorch >= 2.1)
LR = 1e-3
UNET_ENCODER = str(CV.get("smp_unet", {}).get("encoder", "resnet34"))
RUN_DIR = RUNS_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(
    f"trainer={TRAINER} epochs={EPOCHS} batch={BATCH} imgsz={IMGSZ} device={DEVICE} resume={RESUME}"
)

## 7а. Обучение YOLOv8-seg (Ultralytics)

`model.train(...)` c `amp=True` (Ultralytics сам отключает AMP вне CUDA), `cache`, `workers`, `imgsz=512`, `patience`
(ранняя остановка), `save_period=1` (чекпоинт каждой эпохи), `seed`/`deterministic`. Лог метрик — `RUN_DIR/results.csv`.
Resume: `YOLO(last.pt).train(resume=True)`. Валидация в процессе обучения идёт по `images/holdout` (`val:` в `data.yaml`).

In [ ]:
# YOLOv8-seg fine-tuning with Ultralytics.
BEST_WEIGHTS: Path | None = None
if TRAINER == "yolo":
    from ultralytics import YOLO

    from geoqc.cv.weights import fetch_weights

    base_weights = REPO_ROOT / CV["weights"]["path"]
    if not base_weights.exists():
        info = fetch_weights(
            CV, update_files=False
        )  # HF checkpoint or COCO fallback (see MODEL_CARD)
        base_weights = REPO_ROOT / info.path
        print("fetched", info.model_version, "fallback" if info.is_fallback else "")
    last_pt = RUN_DIR / "weights" / "last.pt"
    if RESUME and last_pt.exists():
        model = YOLO(str(last_pt))
        results = model.train(resume=True)
    else:
        model = YOLO(str(base_weights))
        results = model.train(
            data=str(DATA_YAML),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            device=DEVICE,
            amp=True,
            cache=CACHE,
            workers=WORKERS,
            patience=PATIENCE,
            project=str(RUNS_DIR),
            name=RUN_NAME,
            exist_ok=True,
            seed=SEED,
            deterministic=True,
            save_period=1,
            plots=False,
            verbose=False,
        )
    BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
    print("best weights:", BEST_WEIGHTS, BEST_WEIGHTS.exists())
    print("metrics log:", RUN_DIR / "results.csv")

## 7б. Обучение U-Net (segmentation_models_pytorch)

Бинарная сегментация: BCE + Dice, AdamW, cosine-LR, аугментации albumentations, AMP (`torch.autocast`, `GradScaler`
на CUDA), опционально `torch.compile`. Чекпоинт каждой эпохи (`last.pt`), лучший по IoU на holdout (`best.pt`), resume,
ранняя остановка, лог `RUN_DIR/log.csv`. Формат чекпоинта совместим с `geoqc.cv.predictor.SmpUnetPredictor`.

In [ ]:
# U-Net trainer (only when TRAINER == "unet").
if TRAINER == "unet":
    import csv
    import time

    import albumentations as alb
    import cv2
    import segmentation_models_pytorch as smp
    from torch.utils.data import DataLoader, Dataset

    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

    class TileMaskDataset(Dataset):
        def __init__(self, root: Path, subset: str, augment: bool) -> None:
            self.images = sorted((root / "images" / subset).glob("*.png"))
            self.masks_dir = root / "masks" / subset
            self.aug = (
                alb.Compose(
                    [
                        alb.HorizontalFlip(p=0.5),
                        alb.VerticalFlip(p=0.5),
                        alb.RandomRotate90(p=0.5),
                        alb.RandomBrightnessContrast(p=0.3),
                    ]
                )
                if augment
                else None
            )

        def __len__(self) -> int:
            return len(self.images)

        def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
            img = cv2.cvtColor(cv2.imread(str(self.images[i]), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
            mask = cv2.imread(str(self.masks_dir / self.images[i].name), cv2.IMREAD_GRAYSCALE)
            if self.aug is not None:
                out = self.aug(image=img, mask=mask)
                img, mask = out["image"], out["mask"]
            x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
            x = (x - torch.tensor(MEAN).view(3, 1, 1)) / torch.tensor(STD).view(3, 1, 1)
            y = torch.from_numpy((mask > 127).astype(np.float32))[None]
            return x, y

    def dice_loss(logits: torch.Tensor, target: torch.Tensor, eps: float = 1.0) -> torch.Tensor:
        prob = torch.sigmoid(logits)
        inter = (prob * target).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        return (1 - (2 * inter + eps) / (denom + eps)).mean()

    @torch.no_grad()
    def holdout_iou(model: torch.nn.Module, loader: DataLoader) -> float:
        model.eval()
        inter = union = 0.0
        for x, y in loader:
            pred = torch.sigmoid(model(x.to(DEVICE))) > 0.5
            tgt = y.to(DEVICE) > 0.5
            inter += (pred & tgt).sum().item()
            union += (pred | tgt).sum().item()
        return inter / union if union else 0.0

    train_loader = DataLoader(
        torch.utils.data.ConcatDataset(
            [TileMaskDataset(d, "train", augment=True) for d in TRAIN_DIRS]
        ),
        batch_size=BATCH,
        shuffle=True,
        num_workers=WORKERS,
        drop_last=False,
    )
    hold_loader = DataLoader(
        TileMaskDataset(DATASET_DIR, "holdout", augment=False),
        batch_size=BATCH,
        shuffle=False,
        num_workers=WORKERS,
    )
    model = smp.Unet(UNET_ENCODER, encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)
    if USE_COMPILE and DEVICE == "cuda":
        model = torch.compile(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    bce = torch.nn.BCEWithLogitsLoss()
    use_amp = DEVICE in {"cuda", "mps"}
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

    last_ckpt, best_ckpt, log_csv = RUN_DIR / "last.pt", RUN_DIR / "best.pt", RUN_DIR / "log.csv"
    start_epoch, best_iou, bad_epochs = 0, -1.0, 0
    if RESUME and last_ckpt.exists():
        state = torch.load(last_ckpt, map_location="cpu", weights_only=False)
        model.load_state_dict(state["state_dict"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        start_epoch, best_iou = state["epoch"] + 1, state.get("best_iou", -1.0)
        print(f"resumed from epoch {start_epoch}, best_iou={best_iou:.4f}")
    if not log_csv.exists() or not RESUME:
        log_csv.write_text("epoch,train_loss,holdout_iou,lr,seconds\n")

    def checkpoint(epoch: int, iou: float) -> dict:
        return {
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "epoch": epoch,
            "best_iou": max(best_iou, iou),
            "arch": "Unet",
            "encoder": UNET_ENCODER,
            "classes": 1,
            "mean": list(MEAN),
            "std": list(STD),
        }

    for epoch in range(start_epoch, EPOCHS):
        t0 = time.time()
        model.train()
        running = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = bce(logits, y) + dice_loss(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * x.shape[0]
        scheduler.step()
        train_loss = running / max(1, len(train_loader.dataset))
        iou = holdout_iou(model, hold_loader)
        torch.save(checkpoint(epoch, iou), last_ckpt)
        if iou > best_iou:
            best_iou, bad_epochs = iou, 0
            torch.save(checkpoint(epoch, iou), best_ckpt)
        else:
            bad_epochs += 1
        with log_csv.open("a") as fh:
            csv.writer(fh).writerow(
                [
                    epoch,
                    f"{train_loss:.5f}",
                    f"{iou:.5f}",
                    f"{scheduler.get_last_lr()[0]:.2e}",
                    f"{time.time() - t0:.1f}",
                ]
            )
        print(
            f"epoch {epoch + 1}/{EPOCHS} loss={train_loss:.4f} holdout_iou={iou:.4f} ({time.time() - t0:.0f}s)"
        )
        if bad_epochs >= PATIENCE:
            print("early stopping")
            break
    BEST_WEIGHTS = best_ckpt
    print("best checkpoint:", BEST_WEIGHTS, f"holdout_iou={best_iou:.4f}")

## 8. Экспорт весов → `model_version`, `config/cv.yaml`, `MODEL_CARD.md`

Лучшие веса копируются в `models/<model_name>.pt`, считается sha256, `model_version = "<model_name>-<sha256[:8]>"`
(контракт §3.6). В `config/cv.yaml` обновляются `backend`, `model_name`, путь к весам, sha256 и `model_version`;
в `models/MODEL_CARD.md` перезаписывается автоблок с гиперпараметрами прогона.

In [ ]:
# Export best weights, compute sha256 -> model_version, update config/cv.yaml and MODEL_CARD.md.
import datetime as dt  # noqa: E402
import shutil  # noqa: E402

from geoqc.cv.weights import (  # noqa: E402
    WeightsInfo,
    model_version_for,
    sha256_of,
    update_cv_config,
    update_model_card,
)

if BEST_WEIGHTS is None or not Path(BEST_WEIGHTS).exists():
    raise RuntimeError("no trained weights found; run the training cell first")
if TRAINER == "yolo":
    MODEL_NAME, BACKEND = "yolov8m-building-seg-ft", "ultralytics"
else:
    MODEL_NAME, BACKEND = f"unet-{UNET_ENCODER}-ft", "smp_unet"
if DATA_SOURCE == "spacenet":
    MODEL_NAME += "-spacenet"  # smoke-test weights never replace the working model
dest = MODELS_DIR / f"{MODEL_NAME}.pt"
if TRAINER == "unet":  # keep only what SmpUnetPredictor needs (drop optimizer state)
    ck = torch.load(BEST_WEIGHTS, map_location="cpu", weights_only=False)
    torch.save(
        {
            k: ck[k]
            for k in (
                "state_dict",
                "arch",
                "encoder",
                "classes",
                "mean",
                "std",
                "epoch",
                "best_iou",
            )
        },
        dest,
    )
else:
    shutil.copyfile(BEST_WEIGHTS, dest)
SHA = sha256_of(dest)
MODEL_VERSION = model_version_for(MODEL_NAME, SHA)
rel_dest = dest.relative_to(REPO_ROOT).as_posix()
updates = {"backend": BACKEND, "model_name": MODEL_NAME, "model_version": MODEL_VERSION}
if BACKEND == "ultralytics":
    updates.update({"weights.path": rel_dest, "weights.sha256": SHA, "weights.is_fallback": False})
else:
    updates.update({"smp_unet.checkpoint": rel_dest, "smp_unet.encoder": UNET_ENCODER})
if DATA_SOURCE == "spacenet":
    print("DATA_SOURCE=spacenet: config/cv.yaml and MODEL_CARD.md are left unchanged")
else:
    update_cv_config(updates, REPO_ROOT / "config" / "cv.yaml")
info = WeightsInfo(
    path=rel_dest,
    sha256=SHA,
    model_name=MODEL_NAME,
    model_version=MODEL_VERSION,
    source_url=f"fine-tuned in notebooks/train_finetune.ipynb from {CV['weights'].get('hf_repo') or CV['weights'].get('fallback_asset')}",
    license="AGPL-3.0 (derivative of Ultralytics weights)"
    if BACKEND == "ultralytics"
    else "MIT (segmentation_models_pytorch) + ImageNet encoder",
    is_fallback=False,
    backend=BACKEND,
    fetched_at=dt.datetime.now(dt.UTC).isoformat(timespec="seconds"),
)
extra = {
    "fine-tune": f"{TRAINER}: epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, patience={PATIENCE}, seed={SEED}, device={DEVICE}, amp=True",
    "train data": f"{DATA_SOURCE}: {N_TRAIN} train tiles; holdout {N_HOLDOUT} tiles excluded"
    + ("; SpaceNet 2 part is CC BY-SA 4.0" if "spacenet" in DATA_SOURCE else ""),
    "run dir": RUN_DIR.relative_to(REPO_ROOT).as_posix(),
}
if DATA_SOURCE != "spacenet":
    update_model_card(info, REPO_ROOT / "models" / "MODEL_CARD.md", extra=extra)
print("model_version =", MODEL_VERSION)
print("weights ->", dest)

## 9. Подбор порога уверенности на holdout-части val

`run_sweep(holdout_only=True)` запускает инференс новыми весами на holdout-тайлах при `sweep.conf_floor`, для каждого порога
из `sweep.thresholds` заново склеивает/фильтрует полигоны и считает P/R/F1 при IoU ≥ 0,5 (формулы `CONTRACTS §4`;
`geoqc.eval.detection_metrics`, если модуль доступен). Лучший порог записывается в `config/cv.yaml → conf_threshold`,
отчёт — `reports/threshold_sweep_val.json`. Тестовый сплит не участвует.

In [ ]:
# Threshold sweep on the holdout subset of val with the freshly exported weights.
from geoqc.cv.sweep import run_sweep  # noqa: E402

report = None
if DATA_SOURCE == "spacenet":
    print("DATA_SOURCE=spacenet: threshold sweep needs the val holdout, skipped")
else:
    CFG = load_config(REPO_ROOT / "config")  # reload: weights/backend were just rewritten
    report = run_sweep(CFG["cv"], CFG, holdout_only=True, device=DEVICE, progress=False)
    print(
        "best:",
        report["best"],
        "| n_gt:",
        report["n_gt"],
        "| tiles:",
        report["n_tiles"],
        "| impl:",
        report["metrics_impl"],
    )
    for row in report["thresholds"]:
        print(
            f"conf={row['conf']:.2f} n={row['n_det']:4d} P={row['precision']:.3f} R={row['recall']:.3f} F1={row['f1']:.3f}"
        )
    print("report:", report["report_path"])

## 10. Что дальше

* **Colab:** скопируйте на Drive `models/<model_name>.pt`, `config/cv.yaml`, `models/MODEL_CARD.md`,
  `reports/threshold_sweep_val.json` (ячейка ниже) и перенесите их в локальный репозиторий.
* **Локально:** `geoqc cv predict --split all --to-db` → `data/interim/detections.geojson` + `buildings_det`;
  затем `geoqc matching run --model-version <model_version>` и `geoqc eval detection --split val` (гейт G3 ≥ 0,70).
* Оценку на тестовом сплите выполняет только Eval Agent, один раз, после фиксации весов и порога.

In [ ]:
# COLAB ONLY: copy the artefacts back to Google Drive.
if IN_COLAB:
    out_dir = Path("/content/drive/MyDrive/geoqc/trained") / MODEL_VERSION
    out_dir.mkdir(parents=True, exist_ok=True)
    artefacts = [dest]
    if report is not None:  # DATA_SOURCE=spacenet leaves config and card untouched
        artefacts += [
            REPO_ROOT / "config" / "cv.yaml",
            REPO_ROOT / "models" / "MODEL_CARD.md",
            Path(report["report_path"]),
        ]
    for src in artefacts:
        shutil.copy2(src, out_dir / Path(src).name)
    if (RUN_DIR / "results.csv").exists():
        shutil.copy2(RUN_DIR / "results.csv", out_dir / "results.csv")
    if (RUN_DIR / "log.csv").exists():
        shutil.copy2(RUN_DIR / "log.csv", out_dir / "log.csv")
    print("saved to", out_dir)